# [LAB-04] 2. 하이퍼파라미터 튜닝

## #01. 준비작업

### 1. 라이브러리 참조

In [1]:
from jussam import load_data
from helpers import *
import datetime as dt
import os
import glob as gl   # 파일 목록을 리스트로 반환하는 파이썬 내장 모듈

# 훈련,검증 데이터 분리 함수
from sklearn.model_selection import train_test_split

# 하이퍼파라미터 튜닝
from sklearn.model_selection import GridSearchCV

📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/
🔖 Version: 0.5.19


In [34]:
# 선형 계열 모델
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier

# 비선형 계열 모델
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# 트리계열 모델
from sklearn.tree import DecisionTreeClassifier

# 앙상블 모델
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

### 2. 데이터 불러오기

- 파생변수를 포함한 최종 변수 채택 후 로그 변환과 라벨링이 완료된 데이터

In [35]:
# 분석용 데이터 전처리가 완료된 데이터 셋
origin = load_data("titanic_features_labelled")
df = my_qtcheck.set_type(origin, as_category=['Title', 'Pclass', 'Embarked', 'HasCabin', 'Survived'])

📚 타이타닉호 침몰 데이터셋의 파생변수 추가 및 범주형에 대한 라벨링을 수행한 버전 (출처: 자체 작업)

    field       description
--  ----------  --------------------------------------------------------------
 0  Title       신분(범주형, Master=0, Miss=1, Mr=2, Mrs=3, Officer=4)
 1  Pclass      티켓 등급(범주형, 1=1등석, 2=2등석, 3=3등석)
 2  Embarked    승선 항구(범주형, C(쉘버그)=0, Q(퀸스타운)=1, S(사우샘프턴)=2)
 3  HasCabin    객실 보유 여부(범주형, 1=보유, 0=미보유)
 4  Age         나이
 5  Fare        요금
 6  FamilySize  가족 규모
 7  Survived    생존 여부 (종속변수,범주형, 1=생존, 0=사망)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   Title       1309 non-null   category
 1   Pclass      1309 non-null   category
 2   Embarked    1309 non-null   category
 3   HasCabin    1309 non-null   category
 4   Age         1309 non-null   float64 
 5   Fare        1309 non-null   float64 
 6   FamilySize  1309 non-null   int64   
 7   Survived    1309 non-

### 3. 독립변수와 종속변수 분리 및 훈련, 검증 데이터 분할

In [36]:
feature = df.drop(columns=["Survived"])
target = df["Survived"]

# 훈련, 검증 데이터 분리
# 분류 문제이므로 stratify=target 을 지정해 훈련/검증의 클래스 비율을 원본과 같게 유지한다
x_train, x_test, y_train, y_test = train_test_split(
    feature, target, test_size=0.2, random_state=RANDOM_STATE, stratify=target)

x_train.shape, x_test.shape, y_train.shape, y_test.shape

((1047, 7), (262, 7), (1047,), (262,))

### 4. 학습을 완료한 모델이 저장될 폴더

In [ ]:
baseline_dir = f"titanic/baseline"  # 베이스라인 모델이 저장되어 있는 경로

if not os.path.exists(baseline_dir):
    os.makedirs(baseline_dir)

tune_dir = f"titanic/tune"          # 하이퍼파라미터 튜닝 모델이 저장될 경로

if not os.path.exists(tune_dir):
    os.makedirs(tune_dir)

## #02. 하이퍼파라미터 튜닝

### 1. LogisticRegression

In [ ]:
%%time

# 로지스틱 회귀의 핵심 하이퍼파라미터는 규제 강도 C 다.
# C 는 규제 강도의 "역수" 라서 값이 작을수록 규제가 강해진다 (Ridge 의 alpha 와 반대 방향).
param_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0],       # 규제 강도의 역수
    "model__solver": ["lbfgs", "liblinear"]   # 최적화 알고리즘
}

model = my_ml.load_model(f"{baseline_dir}/logistic.pkl")    # 베이스모델 로드

# 파라미터 탐색을 위한 그리드 서치 객체 생성
gs = GridSearchCV(estimator=model, param_grid=param_grid,
    cv=5, scoring="f1", n_jobs=-1)

gs.fit(x_train, y_train)                            # 파라미터 탐색 수행
gs.name_ = f"{model.name_}_tuned"                   # 튜닝 완료 객체에 이름 부여
my_ml.save_model(gs, f"{tune_dir}/logistic.pkl")    # 튜닝된 모델 저장
display(my_ml.cls_score(gs, x_test, y_test))        # 성능 평가 결과 출력
display(gs)                                         # 학습 모형 구조 출력

,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,LogLoss,MCC
Model,,,,,,,,
LogisticRegression,0.859,0.837,0.778,0.806,0.902,0.873,0.363,0.697


CPU times: user 67.7 ms, sys: 10.4 ms, total: 78.1 ms
Wall time: 2.2 s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...state=3217))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.01, 0.1, ...], 'model__solver': ['lbfgs', 'liblinear']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also dis

### 2. RidgeClassifier

In [ ]:
%%time
# [실제 탐색용] 20개 조합
# param_grid = {
#     "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
#     "model__solver": ["auto", "svd", "cholesky", "lsqr"]
# }
# [수업용] 8개 조합. alpha 가 클수록 계수를 0 쪽으로 강하게 눌러 분산을 줄인다.
param_grid = {
    "model__alpha": [0.1, 1.0, 10.0, 100.0],        # 규제 강도
    "model__solver": ["auto", "lsqr"]               # 최적화 알고리즘
}

model = my_ml.load_model(f"{baseline_dir}/ridge.pkl")   # 베이스모델 로드

# 파라미터 탐색을 위한 그리드 서치 객체 생성
gs = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring="f1", n_jobs=-1)
gs.fit(x_train, y_train)                            # 파라미터 탐색 수행
gs.name_ = f"{model.name_}_tuned"                   # 튜닝 완료 객체에 이름 부여
my_ml.save_model(gs, f"{tune_dir}/ridge.pkl")       # 튜닝된 모델 저장
display(my_ml.cls_score(gs, x_test, y_test))        # 성능 평가 결과 출력
display(gs)                                         # 학습 모형 구조 출력

,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,LogLoss,MCC
Model,,,,,,,,
RidgeClassifier,0.847,0.817,0.768,0.792,0.907,0.885,NaN,0.672


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...state=3217))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__alpha': [0.1, 1.0, ...], 'model__solver': ['auto', 'lsqr']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displa

CPU times: user 106 ms, sys: 21.3 ms, total: 127 ms
Wall time: 241 ms


### 3. Lasso

In [ ]:
%%time

# [실제 탐색용] 15개 조합
# param_grid = {
#     "model__C": [0.001, 0.01, 0.1, 1.0, 10.0],
#     "model__solver": ["liblinear"]
# }
# [수업용] 8개 조합.
param_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0],       # 규제 강도의 역수 (L1)
    "model__solver": ["liblinear"]            # L1 을 지원하는 최적화 알고리즘
}

model = my_ml.load_model(f"{baseline_dir}/lasso.pkl")   # 베이스모델 로드

# 파라미터 탐색을 위한 그리드 서치 객체 생성
gs = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring="f1", n_jobs=-1)
gs.fit(x_train, y_train)                            # 파라미터 탐색 수행
gs.name_ = f"{model.name_}_tuned"                   # 튜닝 완료 객체에 이름 부여
my_ml.save_model(gs, f"{tune_dir}/lasso.pkl")       # 튜닝된 모델 저장
display(my_ml.cls_score(gs, x_test, y_test))        # 성능 평가 결과 출력
display(gs)                                         # 학습 모형 구조 출력

,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,LogLoss,MCC
Model,,,,,,,,
LogisticRegression,0.859,0.837,0.778,0.806,0.902,0.873,0.363,0.697


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...liblinear'))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.01, 0.1, ...], 'model__solver': ['liblinear']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- 

CPU times: user 88.8 ms, sys: 15.6 ms, total: 104 ms
Wall time: 203 ms


### 4. ElasticNet

In [ ]:
%%time

# [실제 탐색용] 20개 조합
# param_grid = {
#     "model__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0],
#     "model__l1_ratio": [0.1, 0.3, 0.5, 0.9]
# }
# [수업용] 8개 조합.
param_grid = {
    "model__alpha": [0.0001, 0.001, 0.01, 0.1],   # 규제 강도
    "model__l1_ratio": [0.2, 0.8]                 # L1 규제의 비중
}

model = my_ml.load_model(f"{baseline_dir}/elasticnet.pkl")   # 베이스모델 로드

# 파라미터 탐색을 위한 그리드 서치 객체 생성
gs = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring="f1", n_jobs=-1)
gs.fit(x_train, y_train)                                # 파라미터 탐색 수행
gs.name_ = f"{model.name_}_tuned"                       # 튜닝 완료 객체에 이름 부여
my_ml.save_model(gs, f"{tune_dir}/elasticnet.pkl")      # 튜닝된 모델 저장
display(my_ml.cls_score(gs, x_test, y_test))            # 성능 평가 결과 출력
display(gs)                                             # 학습 모형 구조 출력

,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,LogLoss,MCC
Model,,,,,,,,
SGDClassifier,0.859,0.837,0.778,0.806,0.901,0.871,0.379,0.697


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...state=3217))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__alpha': [0.0001, 0.001, ...], 'model__l1_ratio': [0.2, 0.8]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displ

CPU times: user 115 ms, sys: 22.9 ms, total: 138 ms
Wall time: 255 ms


### 5. KNN

In [ ]:
%%time

# [실제 탐색용] 24개 조합
# param_grid = {
#     "model__n_neighbors": [3, 5, 10, 20, 30, 50],
#     "model__weights": ["uniform", "distance"],
#     "model__p": [1, 2]
# }
# [수업용] 8개 조합. 이웃 수가 적으면 과대적합, 많으면 과소적합 쪽으로 기운다.
param_grid = {
    "model__n_neighbors": [5, 10, 20, 30],       # 참조할 이웃 수
    "model__weights": ["uniform", "distance"]    # 거리에 따른 가중 방식
}

model = my_ml.load_model(f"{baseline_dir}/kneighbors.pkl")   # 베이스모델 로드

# 파라미터 탐색을 위한 그리드 서치 객체 생성
gs = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring="f1", n_jobs=-1)
gs.fit(x_train, y_train)                                # 파라미터 탐색 수행
gs.name_ = f"{model.name_}_tuned"                       # 튜닝 완료 객체에 이름 부여
my_ml.save_model(gs, f"{tune_dir}/kneighbors.pkl")      # 튜닝된 모델 저장
display(my_ml.cls_score(gs, x_test, y_test))            # 성능 평가 결과 출력
display(gs)                                             # 학습 모형 구조 출력

,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,LogLoss,MCC
Model,,,,,,,,
KNeighborsClassifier,0.855,0.835,0.768,0.800,0.900,0.842,0.386,0.688


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...(n_jobs=-1))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__n_neighbors': [5, 10, ...], 'model__weights': ['uniform', 'distance']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is 

CPU times: user 121 ms, sys: 27.1 ms, total: 148 ms
Wall time: 299 ms


### 6. Support Vector Machine

In [ ]:
%%time

# [실제 탐색용] 18개 조합 --> 표본 수의 제곱에 비례해 학습 시간이 늘어나므로 매우 오래 걸린다
# param_grid = {
#     "model__C": [0.1, 1.0, 10.0],
#     "model__gamma": ["scale", 0.01, 0.1],
#     "model__kernel": ["rbf", "poly"]
# }
# [수업용] SVC 는 확률 계산 때문에 이 노트북에서 가장 느리므로 튜닝 시간을 줄이기 위해 조합을 2개로 축소한다.
param_grid = {
    "model__C": [1.0],              # 오차 허용에 대한 벌점 (클수록 훈련 데이터에 밀착)
    "model__gamma": ["scale", 0.01] # RBF 커널의 영향 반경
}

model = my_ml.load_model(f"{baseline_dir}/svc.pkl")   # 베이스모델 로드

# 파라미터 탐색을 위한 그리드 서치 객체 생성
gs = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring="f1", n_jobs=-1)
gs.fit(x_train, y_train)                            # 파라미터 탐색 수행
gs.name_ = f"{model.name_}_tuned"                   # 튜닝 완료 객체에 이름 부여
my_ml.save_model(gs, f"{tune_dir}/svc.pkl")         # 튜닝된 모델 저장
display(my_ml.cls_score(gs, x_test, y_test))        # 성능 평가 결과 출력
display(gs)                                         # 학습 모형 구조 출력

,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,LogLoss,MCC
Model,,,,,,,,
SVC,0.863,0.846,0.778,0.811,0.859,0.841,0.382,0.705


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...state=3217))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [1.0], 'model__gamma': ['scale', 0.01]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the f

CPU times: user 137 ms, sys: 16.5 ms, total: 154 ms
Wall time: 275 ms


### 7. Decision Tree

In [ ]:
%%time

# [실제 탐색용] 24개 조합
# param_grid = {
#     "model__max_depth": [4, 6, 10, 14, 20, None],
#     "model__min_samples_leaf": [1, 5, 10, 20]
# }
# [수업용] 8개 조합.
param_grid = {
    "model__max_depth": [6, 10, 14, None],   # 트리의 최대 깊이
    "model__min_samples_leaf": [1, 10]       # 잎 노드가 가져야 할 최소 표본 수
}

model = my_ml.load_model(f"{baseline_dir}/decisiontree.pkl")   # 베이스모델 로드

# 파라미터 탐색을 위한 그리드 서치 객체 생성
gs = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring="f1", n_jobs=-1)
gs.fit(x_train, y_train)                                # 파라미터 탐색 수행
gs.name_ = f"{model.name_}_tuned"                       # 튜닝 완료 객체에 이름 부여
my_ml.save_model(gs, f"{tune_dir}/decisiontree.pkl")    # 튜닝된 모델 저장
display(my_ml.cls_score(gs, x_test, y_test))            # 성능 평가 결과 출력
display(gs)                                             # 학습 모형 구조 출력

,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,LogLoss,MCC
Model,,,,,,,,
DecisionTreeClassifier,0.866,0.864,0.768,0.813,0.875,0.838,1.135,0.713


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...state=3217))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [6, 10, ...], 'model__min_samples_leaf': [1, 10]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also disp

CPU times: user 97.8 ms, sys: 17 ms, total: 115 ms
Wall time: 229 ms


### 8. Random Forest

In [ ]:
%%time

# [실제 탐색용] 18개 조합
# param_grid = {
#     "model__n_estimators": [100, 300, 500],
#     "model__max_depth": [10, 20, None],
#     "model__min_samples_leaf": [1, 5]
# }
# [수업용] 트리를 n_estimators 개 만큼 학습하므로 조합 하나당 성능 평가 시간이 오래 걸린다 --> 4개로 축소
param_grid = {
    "model__n_estimators": [100, 300],     # 숲을 이루는 트리 개수
    "model__min_samples_leaf": [1, 5]      # 잎 노드가 가져야 할 최소 표본 수
}

model = my_ml.load_model(f"{baseline_dir}/randomforest.pkl")   # 베이스모델 로드

# 파라미터 탐색을 위한 그리드 서치 객체 생성
gs = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring="f1", n_jobs=-1)
gs.fit(x_train, y_train)                                # 파라미터 탐색 수행
gs.name_ = f"{model.name_}_tuned"                       # 튜닝 완료 객체에 이름 부여
my_ml.save_model(gs, f"{tune_dir}/randomforest.pkl")    # 튜닝된 모델 저장
display(my_ml.cls_score(gs, x_test, y_test))            # 성능 평가 결과 출력
display(gs)                                             # 학습 모형 구조 출력

,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,LogLoss,MCC
Model,,,,,,,,
RandomForestClassifier,0.863,0.854,0.768,0.809,0.912,0.889,0.350,0.704


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...state=3217))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__min_samples_leaf': [1, 5], 'model__n_estimators': [100, 300]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also disp

CPU times: user 225 ms, sys: 71.4 ms, total: 296 ms
Wall time: 1.11 s


### 9. XGBoost

In [ ]:
%%time

# [실제 탐색용] 27개 조합
# param_grid = {
#     "model__n_estimators": [300, 600, 1000],
#     "model__max_depth": [3, 4, 6],
#     "model__learning_rate": [0.03, 0.05, 0.1]
# }
# [수업용] 8개 조합.
param_grid = {
    "model__n_estimators": [300, 600],        # 부스팅 라운드 수
    "model__max_depth": [4, 6],               # 트리 깊이
    "model__learning_rate": [0.05, 0.1]       # 각 트리의 반영 비율
}

model = my_ml.load_model(f"{baseline_dir}/xgb.pkl")   # 베이스모델 로드

# 파라미터 탐색을 위한 그리드 서치 객체 생성
gs = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring="f1", n_jobs=-1)
gs.fit(x_train, y_train)                            # 파라미터 탐색 수행
gs.name_ = f"{model.name_}_tuned"                   # 튜닝 완료 객체에 이름 부여
my_ml.save_model(gs, f"{tune_dir}/xgb.pkl")         # 튜닝된 모델 저장
display(my_ml.cls_score(gs, x_test, y_test))        # 성능 평가 결과 출력
display(gs)                                         # 학습 모형 구조 출력

,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,LogLoss,MCC
Model,,,,,,,,
XGBClassifier,0.851,0.841,0.747,0.791,0.903,0.887,0.364,0.679


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__learning_rate': [0.05, 0.1], 'model__max_depth': [4, 6], 'model__n_estimators': [300, 600]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is display

CPU times: user 695 ms, sys: 7.49 s, total: 8.18 s
Wall time: 1.39 s


### 10.  LightGBM

In [ ]:
%%time

# [실제 탐색용] 27개 조합
# param_grid = {
#     "model__n_estimators": [300, 600, 1000],
#     "model__num_leaves": [15, 31, 63],
#     "model__learning_rate": [0.03, 0.05, 0.1]
# }
# [수업용] 8개 조합. LightGBM 은 깊이 대신 잎 개수(num_leaves)로 복잡도를 조절한다.
param_grid = {
    "model__n_estimators": [300, 600],        # 부스팅 라운드 수
    "model__num_leaves": [31, 63],            # 트리 하나가 가질 잎 개수
    "model__learning_rate": [0.05, 0.1]       # 각 트리의 반영 비율
}

model = my_ml.load_model(f"{baseline_dir}/lgbm.pkl")   # 베이스모델 로드

# 파라미터 탐색을 위한 그리드 서치 객체 생성
gs = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring="f1", n_jobs=-1)
gs.fit(x_train, y_train)                            # 파라미터 탐색 수행
gs.name_ = f"{model.name_}_tuned"                   # 튜닝 완료 객체에 이름 부여
my_ml.save_model(gs, f"{tune_dir}/lgbm.pkl")        # 튜닝된 모델 저장
display(my_ml.cls_score(gs, x_test, y_test))        # 성능 평가 결과 출력
display(gs)                                         # 학습 모형 구조 출력

,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,LogLoss,MCC
Model,,,,,,,,
LGBMClassifier,0.821,0.810,0.687,0.743,0.902,0.879,0.428,0.612


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...verbose=-1))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__learning_rate': [0.05, 0.1], 'model__n_estimators': [300, 600], 'model__num_leaves': [31, 63]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displa

CPU times: user 2.98 s, sys: 38.2 s, total: 41.2 s
Wall time: 4min 47s


### 11. CatBoost

In [ ]:
%%time
# [실제 탐색용] 36개 조합 x 5-fold = CatBoost 학습 180회
# param_grid = {
#     "model__iterations": [500, 1000, 2000],
#     "model__depth": [4, 6, 8, 10],
#     "model__learning_rate": [0.03, 0.05, 0.1]
# }
# [수업용] 각 하이퍼파라미터를 일부만 두어 조합을 축소한다.
param_grid = {
    "model__iterations": [500, 1000],
    "model__depth": [4, 6, 8],
    "model__learning_rate": [0.03, 0.05, 0.1]
}

model = my_ml.load_model(f"{baseline_dir}/catboost.pkl") # 베이스모델 로드

# 파라미터 탐색을 위한 그리드 서치 객체 생성
gs = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring="f1", n_jobs=-1)

# 파라미터 탐색 수행 --> 카테고리타입 지정해야 함
gs.fit(x_train, y_train, model__cat_features=my_qtcheck.get_categorical_column_names(x_train))
gs.name_ = f"{model.name_}_tuned"                   # 튜닝 완료 객체에 이름 부여
my_ml.save_model(gs, f"{tune_dir}/catboost.pkl")    # 튜닝된 모델 저장
display(my_ml.cls_score(gs, x_test, y_test))        # 성능 평가 결과 출력
display(gs)                                         # 학습 모형 구조 출력

/Users/leekh/.pyenv/versions/3.13.9/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,LogLoss,MCC
Model,,,,,,,,
CatBoostClassifier,0.844,0.892,0.667,0.763,0.905,0.887,0.377,0.665


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step... verbose=0))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__depth': [4, 6, ...], 'model__iterations': [500, 1000], 'model__learning_rate': [0.03, 0.05, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is dis

CPU times: user 6.31 s, sys: 9.62 s, total: 15.9 s
Wall time: 30 s


## #03. 최고 성능 모델 선정

### 1. 튜닝 모델 파일 목록

In [31]:
# 작업폴더 내의 모든 pkl 파일 목록 확인
model_pickles = gl.glob(f"{tune_dir}/*.pkl")
print(model_pickles)

['titanic/tune/logistic.pkl', 'titanic/tune/kneighbors.pkl', 'titanic/tune/catboost.pkl', 'titanic/tune/elasticnet.pkl', 'titanic/tune/lasso.pkl', 'titanic/tune/ridge.pkl', 'titanic/tune/randomforest.pkl', 'titanic/tune/svc.pkl', 'titanic/tune/xgb.pkl', 'titanic/tune/decisiontree.pkl', 'titanic/tune/lgbm.pkl']


### 2. 튜닝 모델 불러오기

In [32]:
models = {}                         # 모델명과 모델 객체를 저장할 딕셔너리

for p in model_pickles:
    model = my_ml.load_model(p)     # 모델 로드
    models[model.name_] = model     # 모델명과 모델 객체를 딕셔너리에 저장

# 로드된 모델과 모델 객체의 타입 출력
for name, model in models.items():
    print(f"- {name}: {type(model)}")

- logistic_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- kneighbors_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- catboost_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- elasticnet_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- lasso_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- ridge_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- randomforest_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- svc_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- xgb_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- decisiontree_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- lgbm_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>


### 3. 모델간 성능평가 비교

In [33]:
my_ml.cls_compare_models(models,                       # 모델 객체들을 담은 딕셔너리
                         x_test,                       # 검증 데이터의 독립 변수
                         y_test,                       # 검증 데이터의 종속 변수
                         primary="F1",                 # 주 지표
                         aux=["ROC_AUC", "Accuracy"])  # 보조 지표


◆ Score Table Ranking : primary='F1', aux=['ROC_AUC', 'Accuracy']

▲ step1: 주 지표(F1) 기준 정렬 — 높을수록 좋음 (DESC)
    1. decisiontree_tuned F1    =            0.813
    2. svc_tuned      F1    =            0.811
    3. randomforest_tuned F1    =            0.809
    4. logistic_tuned F1    =            0.806
    5. elasticnet_tuned F1    =            0.806
    6. lasso_tuned    F1    =            0.806
    7. kneighbors_tuned F1    =            0.800
    8. ridge_tuned    F1    =            0.792
    9. xgb_tuned      F1    =            0.791
   10. catboost_tuned F1    =            0.763
   11. lgbm_tuned     F1    =            0.743

▲ step2: 근소 격차 그룹 묶기 (1등의 5% 이내)
   - 1등 F1     : 0.813
   - 허용 범위    : F1 ≥ 0.772
   - 근소 격차 그룹 (9) : ['decisiontree_tuned', 'svc_tuned', 'randomforest_tuned', 'logistic_tuned', 'elasticnet_tuned', 'lasso_tuned', 'kneighbors_tuned', 'ridge_tuned', 'xgb_tuned']
   - 그룹 외부     (2) : ['catboost_tuned', 'lgbm_tuned']

▲ step3: 보조 지표 결정적 결함 점검 (근소 격차 그룹 내부)
   - 

,Rank,Group,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,LogLoss,MCC,F1_Gap
name,,,,,,,,,,,,
decisiontree_tuned,1,Contender,DecisionTreeClassifier,0.866,0.864,0.768,0.813,0.875,0.838,1.135,0.713,-0.000
randomforest_tuned,2,Contender,RandomForestClassifier,0.863,0.854,0.768,0.809,0.912,0.889,0.350,0.704,0.005
logistic_tuned,3,Contender,LogisticRegression,0.859,0.837,0.778,0.806,0.902,0.873,0.363,0.697,0.008
elasticnet_tuned,4,Contender,SGDClassifier,0.859,0.837,0.778,0.806,0.901,0.871,0.379,0.697,0.008
lasso_tuned,5,Contender,LogisticRegression,0.859,0.837,0.778,0.806,0.902,0.873,0.363,0.697,0.008
kneighbors_tuned,6,Contender,KNeighborsClassifier,0.855,0.835,0.768,0.800,0.900,0.842,0.386,0.688,0.016
ridge_tuned,7,Contender,RidgeClassifier,0.847,0.817,0.768,0.792,0.907,0.885,NaN,0.672,0.026
xgb_tuned,8,Contender,XGBClassifier,0.851,0.841,0.747,0.791,0.903,0.887,0.364,0.679,0.026
svc_tuned,9,Contender,SVC,0.863,0.846,0.778,0.811,0.859,0.841,0.382,0.705,0.003
